In [0]:
storage_account = "insurancestorage01"

spark.conf.set(
"fs.azure.account.key."+storage_account+".dfs.core.windows.net",
"azure_Storage_key"
)

spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

bronze_adls_path = "abfss://bronze@"+storage_account+".dfs.core.windows.net/claims/"
silver_adls_path = "abfss://silver@"+storage_account+".dfs.core.windows.net/claims/"

# SCHEMA ENFORCEMENT

In [0]:
from pyspark.sql.types import *

claims_schema = StructType([

    StructField("claim_id", IntegerType(), False),
    StructField("claim_number", StringType(), False),
    StructField("customer_policy_id", IntegerType(), False),
    StructField("hospital_id", IntegerType(), False),

    StructField("claim_date", StringType(), True),
    StructField("hospitalization_date", StringType(), True),
    StructField("discharge_date", StringType(), True),

    StructField("diagnosis_code", StringType(), True),
    StructField("diagnosis_description", StringType(), True),
    StructField("room_type", StringType(), True),

    StructField("total_bill", DoubleType(), True),
    StructField("claimed_amount", DoubleType(), True),
    StructField("approved_amount", DoubleType(), True),
    StructField("rejected_amount", DoubleType(), True),

    StructField("claim_status", StringType(), True),
    StructField("rejection_reason", StringType(), True),
    StructField("settlement_date", StringType(), True),
    StructField("processing_days", IntegerType(), True),
    StructField("created_at", TimestampType(), True)

])

In [0]:
claims_df = (spark.read
             .format("parquet")
            #  .schema(claims_schema)
            .option("recursiveFileLookup", "true")
             .load(bronze_adls_path))
display(claims_df)

claim_id,claim_number,customer_policy_id,hospital_id,claim_date,hospitalization_date,discharge_date,diagnosis_code,diagnosis_description,room_type,total_bill_amount,claimed_amount,approved_amount,rejected_amount,claim_status,rejection_reason,settlement_date,processing_days,created_at
1,CLM-00000001,1,239,2025-05-23T00:00:00Z,2025-05-22T00:00:00Z,2025-05-27T00:00:00Z,K35,Appendicitis,Semi-Private,51129.510000000000000000,51129.510000000000000000,51129.510000000000000000,0E-18,Approved,null,2025-06-01T00:00:00Z,9,2025-06-03T20:12:24Z
2,CLM-00000002,2,500,2024-10-15T00:00:00Z,2024-10-11T00:00:00Z,2024-10-19T00:00:00Z,I21,Heart Attack,Semi-Private,349753.830000000000000000,349753.830000000000000000,349753.830000000000000000,0E-18,Approved,null,2024-10-29T00:00:00Z,14,2025-05-07T07:05:18Z
3,CLM-00000003,3,453,2024-03-29T00:00:00Z,2024-03-24T00:00:00Z,2024-04-03T00:00:00Z,J18,Pneumonia,Private,433481.660000000000000000,433481.660000000000000000,301490.590000000000000000,131991.070000000000000000,Partially Approved,Policy Sub-limit,2024-04-04T00:00:00Z,6,2025-05-20T09:50:09Z
4,CLM-00000004,4,133,2024-08-22T00:00:00Z,2024-08-19T00:00:00Z,2024-08-29T00:00:00Z,K35,Appendicitis,Semi-Private,264849.150000000000000000,264849.150000000000000000,0E-18,264849.150000000000000000,Rejected,Pre-existing condition not covered,null,8,2026-01-16T14:36:18Z
5,CLM-00000005,5,437,2023-12-18T00:00:00Z,2023-12-14T00:00:00Z,2023-12-17T00:00:00Z,J18,Pneumonia,General,47486.710000000000000000,47486.710000000000000000,47486.710000000000000000,0E-18,Approved,null,2023-12-28T00:00:00Z,10,2023-12-26T10:27:34Z
6,CLM-00000006,6,444,2025-01-08T00:00:00Z,2025-01-03T00:00:00Z,2025-01-09T00:00:00Z,J18,Pneumonia,General,152045.710000000000000000,152045.710000000000000000,0E-18,152045.710000000000000000,Rejected,Policy expired,null,14,2025-05-13T20:51:52Z
7,CLM-00000007,7,363,2025-03-30T00:00:00Z,2025-03-28T00:00:00Z,2025-04-07T00:00:00Z,I21,Heart Attack,Semi-Private,337409.340000000000000000,337409.340000000000000000,337409.340000000000000000,0E-18,Approved,null,2025-04-03T00:00:00Z,4,2025-07-13T18:29:13Z
8,CLM-00000008,8,358,2023-12-13T00:00:00Z,2023-12-09T00:00:00Z,2023-12-17T00:00:00Z,N20,Kidney Stone,General,268492.050000000000000000,268492.050000000000000000,268492.050000000000000000,0E-18,Approved,null,2023-12-26T00:00:00Z,13,2024-10-25T17:26:06Z
9,CLM-00000009,9,222,2023-04-14T00:00:00Z,2023-04-12T00:00:00Z,2023-04-15T00:00:00Z,I21,Heart Attack,General,163427.120000000000000000,163427.120000000000000000,163427.120000000000000000,0E-18,Approved,null,2023-04-24T00:00:00Z,10,2024-05-11T04:04:57Z
10,CLM-00000010,10,24,2025-03-06T00:00:00Z,2025-03-04T00:00:00Z,2025-03-10T00:00:00Z,I21,Heart Attack,Semi-Private,157598.000000000000000000,157598.000000000000000000,157598.000000000000000000,0E-18,Approved,null,2025-03-11T00:00:00Z,5,2025-09-14T21:55:04Z


# SILVER TRANSFORMATION

In [0]:
from pyspark.sql.functions import *
# claims_df.printSchema()
claims_df_silver = (
    claims_df
    .drop("customer_policy_id") #this column is not needed.

    # Changing datetime formats to date (will be applied when if they are in string)
    .withColumn("claim_date", to_date(date_format("claim_date", "yyyy-MM-dd")))
    .withColumn("hospitalization_date", to_date(date_format("hospitalization_date", "yyyy-MM-dd")))
    .withColumn("discharge_date", to_date(date_format("discharge_date", "yyyy-MM-dd")))
    .withColumn("settlement_date", to_date(date_format("settlement_date", "yyyy-MM-dd")))
    .withColumn("created_at", to_date(date_format("created_at", "yyyy-MM-dd")))

    # Data normalisation
    .withColumn("total_bill_amount", round(col("total_bill_amount"), 2))
    .withColumn("claimed_amount", round(col("claimed_amount").cast("double"),2))
    .withColumn("approved_amount", round(col("approved_amount").cast("double"),2))
    .withColumn("rejected_amount", round(col("rejected_amount").cast("double"),2))

    #Handling nulls
    .withColumn("rejection_reason",
                when(
                    (col("rejection_reason").isNull()) & 
                    ((lower(col("claim_status"))=="approved") | (lower(col("claim_status"))=="under investigation"))
                    , "NA")
                .otherwise(col("rejection_reason"))
    )

    .withColumn("settlement_date",
                when(
                    (col("settlement_date").isNull()) & (lower(col("claim_status"))=="approved"), col("discharge_date")
                )
                .otherwise(col("settlement_date"))
                )

    #Derived columns
    .withColumn("length_of_stay", datediff(col("discharge_date"), col("hospitalization_date")))
    .withColumn("approval_rate", round((col("approved_amount"))*100/(col("total_bill_amount")),2))
    .withColumn("rejection_rate", round((col("rejected_amount"))*100/(col("total_bill_amount")),2))
    .withColumn("claim_year", year("claim_date"))
    .withColumn("claim_month", month("claim_date"))

    #Data Validate
    .filter(col("discharge_date")>= col("hospitalization_date"))
    .filter(col("approved_amount")<=col("claimed_amount"))

    .dropDuplicates(["claim_id"])
)
display(claims_df_silver)

claim_id,claim_number,hospital_id,claim_date,hospitalization_date,discharge_date,diagnosis_code,diagnosis_description,room_type,total_bill_amount,claimed_amount,approved_amount,rejected_amount,claim_status,rejection_reason,settlement_date,processing_days,created_at,length_of_stay,approval_rate,rejection_rate,claim_year,claim_month
1,CLM-00000001,239,2025-05-23,2025-05-22,2025-05-27,K35,Appendicitis,Semi-Private,51129.51,51129.51,51129.51,0.0,Approved,NA,2025-06-01,9,2025-06-03,5,100.0,0.0,2025,5
2,CLM-00000002,500,2024-10-15,2024-10-11,2024-10-19,I21,Heart Attack,Semi-Private,349753.83,349753.83,349753.83,0.0,Approved,NA,2024-10-29,14,2025-05-07,8,100.0,0.0,2024,10
3,CLM-00000003,453,2024-03-29,2024-03-24,2024-04-03,J18,Pneumonia,Private,433481.66,433481.66,301490.59,131991.07,Partially Approved,Policy Sub-limit,2024-04-04,6,2025-05-20,10,69.55,30.45,2024,3
4,CLM-00000004,133,2024-08-22,2024-08-19,2024-08-29,K35,Appendicitis,Semi-Private,264849.15,264849.15,0.0,264849.15,Rejected,Pre-existing condition not covered,null,8,2026-01-16,10,0.0,100.0,2024,8
5,CLM-00000005,437,2023-12-18,2023-12-14,2023-12-17,J18,Pneumonia,General,47486.71,47486.71,47486.71,0.0,Approved,NA,2023-12-28,10,2023-12-26,3,100.0,0.0,2023,12
6,CLM-00000006,444,2025-01-08,2025-01-03,2025-01-09,J18,Pneumonia,General,152045.71,152045.71,0.0,152045.71,Rejected,Policy expired,null,14,2025-05-13,6,0.0,100.0,2025,1
7,CLM-00000007,363,2025-03-30,2025-03-28,2025-04-07,I21,Heart Attack,Semi-Private,337409.34,337409.34,337409.34,0.0,Approved,NA,2025-04-03,4,2025-07-13,10,100.0,0.0,2025,3
8,CLM-00000008,358,2023-12-13,2023-12-09,2023-12-17,N20,Kidney Stone,General,268492.05,268492.05,268492.05,0.0,Approved,NA,2023-12-26,13,2024-10-25,8,100.0,0.0,2023,12
9,CLM-00000009,222,2023-04-14,2023-04-12,2023-04-15,I21,Heart Attack,General,163427.12,163427.12,163427.12,0.0,Approved,NA,2023-04-24,10,2024-05-11,3,100.0,0.0,2023,4
10,CLM-00000010,24,2025-03-06,2025-03-04,2025-03-10,I21,Heart Attack,Semi-Private,157598.00,157598.0,157598.0,0.0,Approved,NA,2025-03-11,5,2025-09-14,6,100.0,0.0,2025,3


In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy(col("claim_number")).orderBy(col("claim_date").desc())

claims_df_silver = (
    claims_df_silver
    .withColumn("rnk", row_number().over(window))
    .filter(col("rnk")==1)
    .drop("rnk")
)

In [0]:
claims_silver = claims_df_silver.select(
    "claim_id",
    "claim_number",
    "hospital_id",
    "claim_date",
    "hospitalization_date",
    "discharge_date",
    "diagnosis_code",
    "diagnosis_description",
    "room_type",
    "length_of_stay",
    "total_bill_amount",
    "claimed_amount",
    "approved_amount",
    "rejected_amount",
    "claim_status",
    "rejection_reason",
    "approval_rate",
    "rejection_rate",
    "settlement_date",
    "processing_days",
    "claim_year",
    "claim_month",
    "created_at",
    
)

claims_silver.write.format("delta").mode("overwrite").partitionBy("claim_year", "claim_month").save(silver_adls_path)
    